# Retrieval-Augmented Generation (RAG) for InvestIQ

---

# 0. Import required libraries
---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q -U langchain-text-splitters sentence-transformers huggingface_hub chromadb torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.

---
# 1. Load the RAG Source Documents

For this project, I will use the 10-K filings of two companies:

- Netflix {{'SimFinId': 103644}, {'Ticker': 'NFLX'}}
- Coca Cola {{'SimFinId': 120376}, {'Ticker': 'cc'}}

These documents will be used as the source material for the RAG pipeline.

In [3]:
NETFLIX_PATH = "/content/drive/MyDrive/data_handling_and_machine_learning/RAG/NFLX/10-K_2026-01-23_0001065280-26-000034.txt"
COCA_COLA_PATH = "/content/drive/MyDrive/data_handling_and_machine_learning/RAG/KO/10-K_2026-02-20_0001628280-26-010047.txt"

In [4]:
# Read both 10-K documents

with open(NETFLIX_PATH, "r", encoding="utf-8") as file:
    netflix_text = file.read()

with open(COCA_COLA_PATH, "r", encoding="utf-8") as file:
    cc_text = file.read()

print("Netflix characters:", len(netflix_text))
print("Coca Cola characters:", len(cc_text))

print("\nNetflix preview:")
print(netflix_text[:500])

print("\nCoca Cola preview:")
print(cc_text[:500])

Netflix characters: 309861
Coca Cola characters: 661405

Netflix preview:
nflx-20251231
0001065280
2025
FY
false
434
428
426
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaap/2025#Liabilities
http://fasb.org/us-gaap/2025#Liabilities
P1Y
http://fasb.org/us-gaap/2025#OtherPensi

Coca Cola preview:
ko-20251231
0000021344
2025
FY
False
http://fasb.org/us-gaap/2025#AssetsOfDisposalGroupIncludingDiscontinuedOperation http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AssetsOfDisposalGroupIncludingDiscontinuedOperation http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AssetsOfDisposalGroupIncludingDiscontinuedOperation http://fasb.org/us-gaap

## 1.1 Inspect the Source Document Structure

The source files are readable, but their beginning contains `SEC/XBRL metadata` and `accounting tags`.

Before building the **RAG** pipeline, I will inspect the documents to determine whether they also contain the narrative text of the $10-K$ filings.

This is important because **RAG** should retrieve meaningful financial and business information rather than primarily retrieving technical `XBRL metadata`.

In [5]:
# Look for common narrative sections in each document

keywords = {
    "business": [
        "Item 1.",
        "ITEM 1.",
    ],
    "risk_factors": [
        "Item 1A.",
        "ITEM 1A.",
    ],
    "md&a": [
        "Item 7.",
        "ITEM 7.",
    ],
    "financial_statements": [
        "Item 8.",
        "ITEM 8.",
    ],
}

for name, text in [
    ("Netflix", netflix_text),
    ("Coca Cola", cc_text)
]:
    print(f"\n===== {name} =====")

    for section, possible_keywords in keywords.items():
        position = -1

        for keyword in possible_keywords:
            position = text.find(keyword)

            if position != -1:
                break

        if position != -1:
            print(f"{section}: found at character {position}")
        else:
            print(f"{section}: not found")


===== Netflix =====
business: found at character 44805
risk_factors: found at character 54015
md&a: found at character 138948
financial_statements: found at character 183310

===== Coca Cola =====
business: found at character 129316
risk_factors: found at character 221517
md&a: found at character 259596
financial_statements: found at character 73701


## 1.2 Inspect a Narrative Section

The source files contain both structured `SEC/XBRL` information and `10-K` narrative content.

I will now inspect a small section of the narrative text to determine how clean and readable the source material is before designing the chunking step.

In [6]:
# Inspect text around the Business section
for name, text in [
    ("Netflix", netflix_text),
    ("Coca Cola", cc_text)
]:
  position = text.lower().find("item 1.")

  print(f"\n==== {name} ====")
  if position != -1:
      print(text[position:position + 3000])
  else:
      print("Business section not found")


==== Netflix ====
Item 1.
Business
ABOUT US
Netflix, Inc. (“Netflix”, the “Company”, “registrant”, “we”, or “us”) is one of the world’s leading entertainment services offering TV series, films, games and live programming across a wide variety of genres and languages. Members can play, pause and resume watching as much as they want, anytime, anywhere, and can change their plans at any time.
Our core strategy is to grow our business globally within the parameters of our operating margin target. We strive to continuously improve our members' experience by offering compelling content that delights them and attracts new members. We aim to offer a range of pricing plans, including our ad-supported subscription plan, to meet a variety of consumer needs. We seek to drive conversation around our content to further enhance member joy, and we are continuously enhancing our user interface to help our members more easily choose content that they will find enjoyable.
BUSINESS SEGMENTS
We operate as

## 1.3 Locate the Main 10-K Narrative

The source files contain both `XBRL metadata` and `narrative filing content`.

The **Netflix file** clearly contains the 10-K table of contents and narrative text. For **Coca Cola**, the first occurrence of *"Business"* was inside XBRL metadata.

I will therefore locate the main `ITEM 1.` section before deciding how to clean the documents.

In [7]:
# Inspect the exact characters around "ITEM 1. Business"

for name, text in [
    ("Netflix", netflix_text),
    ("Coca Cola", cc_text)
]:
    print(f"\n========== {name} ==========")

    position = text.find("ITEM")

    print("First 'ITEM' found at:", position)

    if position != -1:
        sample = text[position:position + 30]

        print("\nVisible text:")
        print(sample)

        print("\nCharacter representations:")
        print(repr(sample))


========== Netflix ==========
First 'ITEM' found at: -1

========== Coca Cola ==========
First 'ITEM' found at: 69218

Visible text:
ITEM 1.  BUSINESS
In this repo

Character representations:
'ITEM 1.\xa0\xa0BUSINESS\nIn this repo'


In [8]:
# Normalize non-breaking spaces

netflix_clean = netflix_text.replace("\xa0", " ")
netflix_clean = netflix_clean.replace("’", "'")
netflix_clean = " ".join(netflix_clean.split())
cc_clean = cc_text.replace("\xa0", " ")
cc_clean = cc_clean.replace("’", "'")
cc_clean = " ".join(cc_clean.split())

print("Netflix:")
print("Before:", len(netflix_text))
print("After: ", len(netflix_clean))

print("\nCoca Cola:")
print("Before:", len(cc_text))
print("After: ", len(cc_clean))

Netflix:
Before: 309861
After:  309179

Coca Cola:
Before: 661405
After:  661251


In [9]:
# Verify that the non-breaking spaces were replaced

print("Netflix:")
print("Raw search:  ", netflix_text.lower().find("item 1."))
print("Clean search:", netflix_clean.lower().find("item 1."))

print("\nCoca Cola:")
print("Raw search:  ", cc_text.lower().find("item 1."))
print("Clean search:", cc_clean.lower().find("item 1."))

Netflix:
Raw search:   44805
Clean search: 40792

Coca Cola:
Raw search:   69218
Clean search: 66078


In [10]:
# Find occurrences of "ITEM 1." in each document
for name, text in [
    ("Netflix", netflix_clean),
    ("Coca Cola", cc_clean)
]:
  print(f"\n==== {name} ====")

  start = 0
  count = 0

  while True:
    position = text.lower().find("item 1.", start)

    if position == -1:
      break

    print(f"\n--- Occurrence {count + 1} at character {position} ---")
    print(text[position:position + 1000])

    start = position + 1
    count += 1


==== Netflix ====

--- Occurrence 1 at character 40792 ---
Item 1. Business 1 Item 1A. Risk Factors 4 Item 1B. Unresolved Staff Comments 15 Item 1C. Cybersecurity 15 Item 2. Properties 17 Item 3. Legal Proceedings 17 Item 4. Mine Safety Disclosures 17 PART II Item 5. Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities 18 Item 6. [Reserved] 19 Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations 20 Item 7A. Quantitative and Qualitative Disclosures About Market Risk 29 Item 8. Financial Statements and Supplementary Data 30 Item 9. Changes in and Disagreements with Accountants on Accounting and Financial Disclosure 30 Item 9A. Controls and Procedures 31 Item 9B. Other Information 33 Item 9C. Disclosure Regarding Foreign Jurisdictions that Prevent Inspections 33 PART III Item 10. Directors, Executive Officers and Corporate Governance 34 Item 11. Executive Compensation 34 Item 12. Security Own

### 1.3.1 Locate the Main 10-K Section Boundaries

The cleaned documents contain multiple sections from the 10-K filing.

Before extracting the narrative content, I will locate the main section headings and their positions in each document.

These positions will help us identify the boundaries between sections such as Business, Risk Factors, Management's Discussion and Analysis, and Financial Statements.

In [11]:
section_headers = [
    "item 1. business",
    "item 1a. risk factors",
    "item 7. management's discussion and analysis",
    "item 8. financial statements",
]

for name, text in [
    ("Netflix", netflix_clean),
    ("Coca Cola", cc_clean)
]:
    print(f"\n========== {name} ==========")

    text_lower = text.lower()
    for header in section_headers:
        print(f"\n{header}")

        start = 0

        while True:
            position = text_lower.find(header, start)

            if position == -1:
                break

            print("  ", position)

            start = position + 1


========== Netflix ==========

item 1. business
   40792
   44792

item 1a. risk factors
   40811
   53965

item 7. management's discussion and analysis
   41135
   138672

item 8. financial statements
   41303
   182904
   196707
   196970

========== Coca Cola ==========

item 1. business
   66078
   69199
   129283
   137743
   178956
   181635
   183394

item 1a. risk factors
   66097
   69028
   124423
   221473
   272826
   277238

item 7. management's discussion and analysis
   66472
   78929
   79271
   259512

item 8. financial statements
   66640
   73681
   82136
   259997
   375176
   609686


### 1.3.2 Inspect Netflix Section Heading Formatting

The Netflix filing uses a different formatting pattern from Coca Cola.

For example, the Business heading is split across two lines:

`ITEM 1.`

`Business`

I will inspect the formatting of several Netflix section headings before creating the extraction logic.

In [12]:
netflix_headers = [
    "ITEM 1.",
    "ITEM 1A.",
    "ITEM 7.",
    "ITEM 8."
]

for header in netflix_headers:
    print(f"\n===== {header} =====")

    start = 0
    count = 0

    while True:
        position = netflix_clean.lower().find(header.lower(), start)

        if position == -1:
            break

        print(f"\nOccurrence {count + 1} at character {position}:")
        print(repr(netflix_clean[position:position + 80]))

        start = position + 1
        count += 1


===== ITEM 1. =====

Occurrence 1 at character 40792:
'Item 1. Business 1 Item 1A. Risk Factors 4 Item 1B. Unresolved Staff Comments 15'

Occurrence 2 at character 44792:
'Item 1. Business ABOUT US Netflix, Inc. (“Netflix”, the “Company”, “registrant”,'

===== ITEM 1A. =====

Occurrence 1 at character 40811:
'Item 1A. Risk Factors 4 Item 1B. Unresolved Staff Comments 15 Item 1C. Cybersecu'

Occurrence 2 at character 53965:
'Item 1A. Risk Factors If any of the following risks actually occur, our business'

===== ITEM 7. =====

Occurrence 1 at character 41135:
"Item 7. Management's Discussion and Analysis of Financial Condition and Results "

Occurrence 2 at character 138672:
"Item 7. Management's Discussion and Analysis of Financial Condition and Results "

===== ITEM 8. =====

Occurrence 1 at character 41303:
'Item 8. Financial Statements and Supplementary Data 30 Item 9. Changes in and Di'

Occurrence 2 at character 182904:
'Item 8. Financial Statements and Supplementary Data The con

---
# 2. Verify the Business Section Boundary

The actual `ITEM 1. Business` section should end when the actual `ITEM 1A. Risk Factors` section begins.

I will inspect the text immediately before and after this boundary for both companies to make sure the section extraction will preserve the correct narrative content.

In [13]:
business_headers = {
    "Netflix": ("item 1. business", "item 1a. risk factors"),
    "Coca Cola": ("item 1. business", "item 1a. risk factors")
}

for name, (business_header, risk_header) in business_headers.items():

    text = netflix_clean if name == "Netflix" else cc_clean
    text_lower = text.lower()

    business_positions = []
    start = 0

    while True:
        position = text_lower.find(business_header, start)

        if position == -1:
            break

        business_positions.append(position)
        start = position + 1

    risk_positions = []
    start = 0

    while True:
        position = text_lower.find(risk_header, start)

        if position == -1:
            break

        risk_positions.append(position)
        start = position + 1

    print(f"\n========== {name} ==========")
    print("Business starts at:", business_positions)
    print("Risk Factors starts at:", risk_positions)


========== Netflix ==========
Business starts at: [40792, 44792]
Risk Factors starts at: [40811, 53965]

========== Coca Cola ==========
Business starts at: [66078, 69199, 129283, 137743, 178956, 181635, 183394]
Risk Factors starts at: [66097, 69028, 124423, 221473, 272826, 277238]


In [14]:
for name, text in [
    ("Netflix", netflix_clean),
    ("Coca Cola", cc_clean)
]:
    text_lower = text.lower()

    business_header = "item 1. business"
    risk_header = "item 1a. risk factors"

    # Find all Business positions
    business_positions = []
    start = 0

    while True:
        position = text_lower.find(business_header, start)

        if position == -1:
            break

        business_positions.append(position)
        start = position + 1

    print(f"\n========== {name} ==========")
    print("Business positions:", business_positions)

    # Inspect each occurrence
    for i, position in enumerate(business_positions):
        print(f"\n--- Business occurrence {i + 1} ---")
        print(text[position:position + 300])


========== Netflix ==========
Business positions: [40792, 44792]

--- Business occurrence 1 ---
Item 1. Business 1 Item 1A. Risk Factors 4 Item 1B. Unresolved Staff Comments 15 Item 1C. Cybersecurity 15 Item 2. Properties 17 Item 3. Legal Proceedings 17 Item 4. Mine Safety Disclosures 17 PART II Item 5. Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Purchases of 

--- Business occurrence 2 ---
Item 1. Business ABOUT US Netflix, Inc. (“Netflix”, the “Company”, “registrant”, “we”, or “us”) is one of the world's leading entertainment services offering TV series, films, games and live programming across a wide variety of genres and languages. Members can play, pause and resume watching as muc

========== Coca Cola ==========
Business positions: [66078, 69199, 129283, 137743, 178956, 181635, 183394]

--- Business occurrence 1 ---
Item 1. Business 2 Item 1A. Risk Factors 12 Item 1B. Unresolved Staff Comments 26 Item 1C. Cybersecurity 26 Item 2. Properties 28 I

In [15]:
# Extract the actual Business section

for name, text in [
    ("Netflix", netflix_clean),
    ("Coca Cola", cc_clean)
]:
    text_lower = text.lower()

    business_header = "item 1. business"
    risk_header = "item 1a. risk factors"

    # Find all Business heading positions
    business_positions = []
    start = 0

    while True:
        position = text_lower.find(business_header, start)

        if position == -1:
            break

        business_positions.append(position)
        start = position + 1

    # Second occurrence = actual Business section
    business_start = business_positions[1]

    # Find the first Risk Factors heading AFTER Business
    risk_start = text_lower.find(
        risk_header,
        business_start
    )

    # Extract Business section
    business_text = text[
        business_start:risk_start
    ]

    print(f"\n========== {name} ==========")
    print("Business starts at:", business_start)
    print("Risk Factors starts at:", risk_start)
    print("Business section length:", len(business_text))

    print("\n--- Start ---")
    print(business_text[:500])

    print("\n--- End ---")
    print(business_text[-500:])


========== Netflix ==========
Business starts at: 44792
Risk Factors starts at: 53965
Business section length: 9173

--- Start ---
Item 1. Business ABOUT US Netflix, Inc. (“Netflix”, the “Company”, “registrant”, “we”, or “us”) is one of the world's leading entertainment services offering TV series, films, games and live programming across a wide variety of genres and languages. Members can play, pause and resume watching as much as they want, anytime, anywhere, and can change their plans at any time. Our core strategy is to grow our business globally within the parameters of our operating margin target. We strive to continu

--- End ---
 and webcasts. We use these channels as well as social media and blogs to communicate with our members and the public about our company, our services and other issues. It is possible that the information we post on social media and blogs could be deemed to be material information. Therefore, we encourage investors, the media, and others interested in o

## 2.1 Define the RAG Section Boundaries

For the RAG baseline, I will extract four sections from each company's 10-K filing:

1. Business
2. Risk Factors
3. Management's Discussion and Analysis (MD&A)
4. Financial Statements and related notes

The section boundaries will be determined from the actual narrative occurrences of the 10-K headings rather than fixed character positions.

The complete Financial Statements section will be preserved, including its tables and notes.

In [16]:
section_patterns = {

    "Business": [
        "item 1. business"
    ],
    "Risk Factors": [
        "item 1a. risk factors"
    ],
    "MD&A": [
        "item 7. management's discussion and analysis"
    ],
    "Financial Statements": [
        "item 8. financial statements"
    ]
}

print("Section patterns defined:")
for section, patterns in section_patterns.items():
    print(f"\n{section}:")
    for pattern in patterns:
        print(repr(pattern))

Section patterns defined:

Business:
'item 1. business'

Risk Factors:
'item 1a. risk factors'

MD&A:
"item 7. management's discussion and analysis"

Financial Statements:
'item 8. financial statements'


## 2.2 Locate the Actual 10-K Section Headings

The same section heading can appear in the Table of Contents and again in the actual filing narrative.

For each section, I will find all matching occurrences and use the last occurrence as the starting position of the actual narrative section.

In [17]:
def find_actual_section_start(text, patterns, occurrence):
    positions = []

    text_lower = text.lower()

    for pattern in patterns:
        pattern_lower = pattern.lower()
        start = 0

        while True:
            position = text_lower.find(
                pattern_lower,
                start
            )

            if position == -1:
                break

            positions.append(position)
            start = position + 1

    positions = sorted(set(positions))

    if len(positions) < occurrence:
        return None

    return positions[occurrence - 1]

section_occurrences = {
    "Netflix": {
        "Business": 2,
        "Risk Factors": 2,
        "MD&A": 2,
        "Financial Statements": 2
    },

    "Coca Cola": {
        "Business": 2,
        "Risk Factors": 3,
        "MD&A": 4,
        "Financial Statements": 5
    }
}

for name, text in [
    ("Netflix", netflix_clean),
    ("Coca Cola", cc_clean)
]:
    print(f"\n========== {name} ==========")

    for section, patterns in section_patterns.items():

        position = find_actual_section_start(
            text,
            patterns,
            section_occurrences[name][section]
        )

        print(f"{section:<22} {position}")


========== Netflix ==========
Business               44792
Risk Factors           53965
MD&A                   138672
Financial Statements   182904

========== Coca Cola ==========
Business               69199
Risk Factors           124423
MD&A                   259512
Financial Statements   375176


---
# 3. Extract the Four RAG Sections

The actual starting positions of the four 10-K sections have been verified.

I will now extract the text belonging to each section:

- Business
- Risk Factors
- Management's Discussion and Analysis (MD&A)
- Financial Statements and related notes

The extracted sections will be stored together with their company and section metadata.

In [18]:
def extract_sections(text, section_patterns, section_occurrences):
    section_starts = {}

    # Find the actual starting position of each section
    for section, patterns in section_patterns.items():
        position = find_actual_section_start(text, patterns, section_occurrences[section])

        if position is None:
            raise ValueError(f"Could not find section: {section}")

        section_starts[section] = position

    # Sort sections by their position in the document
    ordered_sections = sorted(
        section_starts.items(),
        key=lambda item: item[1]
    )

    extracted = {}

    # Extract each section up to the beginning of the next section
    for i, (section, start) in enumerate(ordered_sections):

        if i < len(ordered_sections) - 1:
            end = ordered_sections[i + 1][1]

        else:
            end = text.lower().find(
                "item 9.",
                start
            )
            if end == -1:
                end = len(text)

        section_text = text[start:end]
        extracted[section] = section_text.strip()

    return extracted


netflix_sections = extract_sections(
    netflix_clean,
    section_patterns,
    section_occurrences["Netflix"]
)

cc_sections = extract_sections(
    cc_clean,
    section_patterns,
    section_occurrences["Coca Cola"]
)

print("Netflix:")
for section, text in netflix_sections.items():
    print(f"{section:<22} {len(text):>8} characters")

print("\nCoca Cola:")
for section, text in cc_sections.items():
    print(f"{section:<22} {len(text):>8} characters")

Netflix:
Business                   9172 characters
Risk Factors              84706 characters
MD&A                      44231 characters
Financial Statements        268 characters

Coca Cola:
Business                  55223 characters
Risk Factors             135088 characters
MD&A                     115663 characters
Financial Statements     233369 characters


## 3.1 Validate the Extracted Sections

Before chunking and embedding the extracted sections, I will inspect their beginnings and endings.

This verifies that each section contains the intended 10-K content and that the section boundaries were applied correctly.

In [19]:
search_term = "netflix, inc. index to financial statements"

position = netflix_clean.lower().find(search_term)

print("Netflix financial statements start:", position)

if position != -1:
    print(
        netflix_clean[position:position + 1000]
    )

Netflix financial statements start: 197187
NETFLIX, INC. INDEX TO FINANCIAL STATEMENTS Page Report of Independent Registered Public Accounting Firm (PCAOB ID: 42 ) 37 Consolidated Statements of Operations 39 Consolidated Statements of Comprehensive Income 40 Consolidated Statements of Cash Flows 41 Consolidated Balance Sheets 42 Consolidated Statements of Stockholders' Equity 43 Notes to Consolidated Financial Statements 44 36 Table of Contents Report of Independent Registered Public Accounting Firm To the Stockholders and the Board of Directors of Netflix, Inc. Opinion on the Financial Statements We have audited the accompanying consolidated balance sheets of Netflix, Inc. (the Company) as of December 31, 2025 and 2024, the related consolidated statements of operations, comprehensive income, stockholders' equity and cash flows for each of the three years in the period ended December 31, 2025, and the related notes (collectively referred to as the “consolidated financial statements"). 

In [20]:
for name, sections in [
    ("Netflix", netflix_sections),
    ("Coca Cola", cc_sections)
]:
    print(f"\n\n========== {name} ==========")

    for section, text in sections.items():
        print(f"\n--- {section}: beginning ---")
        print(text[:300])

        print(f"\n--- {section}: ending ---")
        print(text[-300:])



========== Netflix ==========

--- Business: beginning ---
Item 1. Business ABOUT US Netflix, Inc. (“Netflix”, the “Company”, “registrant”, “we”, or “us”) is one of the world's leading entertainment services offering TV series, films, games and live programming across a wide variety of genres and languages. Members can play, pause and resume watching as muc

--- Business: ending ---
tion we post on social media and blogs could be deemed to be material information. Therefore, we encourage investors, the media, and others interested in our company to review the information we post on the social media channels and blogs listed on our investor relations website. 3 Table of Contents

--- Risk Factors: beginning ---
Item 1A. Risk Factors If any of the following risks actually occur, our business, financial condition and results of operations could be harmed. In that case, the trading price of our common stock could decline, and you could lose all or part of your investment. Risks Related t

# Change netflix statements section

In [21]:
# Find the actual Netflix financial statements

netflix_financial_start = netflix_clean.lower().find(
    "netflix, inc. index to financial statements"
)

signature_position = netflix_clean.lower().find(
    "signatures",
    netflix_financial_start
)

if netflix_financial_start == -1:
    raise ValueError(
        "Could not find Netflix financial statements"
    )

# Replace the short Item 8 section with the actual financial statements
netflix_sections["Financial Statements"] = (
    netflix_clean[netflix_financial_start:signature_position].strip()
)

print(
    "Netflix Financial Statements:",
    len(netflix_sections["Financial Statements"]),
    "characters"
)

print("\n--- Beginning ---")
print(
    netflix_sections["Financial Statements"][:500]
)

print("\n--- Ending ---")
print(
    netflix_sections["Financial Statements"][-500:]
)

Netflix Financial Statements: 109081 characters

--- Beginning ---
NETFLIX, INC. INDEX TO FINANCIAL STATEMENTS Page Report of Independent Registered Public Accounting Firm (PCAOB ID: 42 ) 37 Consolidated Statements of Operations 39 Consolidated Statements of Comprehensive Income 40 Consolidated Statements of Cash Flows 41 Consolidated Balance Sheets 42 Consolidated Statements of Stockholders' Equity 43 Notes to Consolidated Financial Statements 44 36 Table of Contents Report of Independent Registered Public Accounting Firm To the Stockholders and the Board of D

--- Ending ---
orated by reference in any filing we make under the Securities Act of 1933 or the Securities Exchange Act of 1934, irrespective of any general incorporation language in any filings. † Indicates a management contract or compensatory plan + Annexes, schedules and/or exhibits have been omitted pursuant to Item 601(a)(5) of Regulation S-K promulgated by the SEC. Netflix agrees to furnish supplementally a copy of any 

### 3.1.1 Verify the RAG Section End Boundaries

The previous extraction showed that using the next selected RAG section as the boundary can accidentally include unrelated 10-K sections.

I will therefore verify the first actual section heading after each of the four RAG sections.

The intended boundaries are:

- Business → ITEM 1A. Risk Factors
- Risk Factors → ITEM 1B. Unresolved Staff Comments
- MD&A → ITEM 7A. Quantitative and Qualitative Disclosures About Market Risk
- Financial Statements → ITEM 9. Changes in and Disagreements With Accountants on Accounting and Financial Disclosure

In [22]:
boundary_patterns = {
    "Business": "item 1a.",
    "Risk Factors": "item 1b.",
    "MD&A": "item 7a.",
    "Financial Statements": "item 9."
}

for name, text in [
    ("Netflix", netflix_clean),
    ("Coca Cola", cc_clean)
]:
    print(f"\n========== {name} ==========")
    text_lower = text.lower()

    for section, boundary in boundary_patterns.items():

        section_start = find_actual_section_start(
            text,
            section_patterns[section],
            section_occurrences[name][section]
        )

        boundary_position = text_lower.find(
            boundary.lower(),
            section_start + 1
        )

        print(f"\n{section}")
        print("Start:", section_start)
        print("End boundary:", boundary_position)

        if section_start is not None and boundary_position != -1:
            print("Section length:", boundary_position - section_start)

            print("\nBoundary text:")
            print(text[boundary_position:boundary_position + 250])


========== Netflix ==========

Business
Start: 44792
End boundary: 53965
Section length: 9173

Boundary text:
Item 1A. Risk Factors If any of the following risks actually occur, our business, financial condition and results of operations could be harmed. In that case, the trading price of our common stock could decline, and you could lose all or part of your

Risk Factors
Start: 53965
End boundary: 129048
Section length: 75083

Boundary text:
Item 1B. Unresolved Staff Comments None. Item 1C. Cybersecurity We have an enterprise-wide information security program designed to identify, protect, detect and respond to and manage reasonably foreseeable cybersecurity risks and threats. To protect

MD&A
Start: 138672
End boundary: 176182
Section length: 37510

Boundary text:
Item 7A. Quantitative and Qualitative Disclosures About Market Risk We are exposed to market risks related to interest rate changes, which affect the market values of our investments and debt, as well as foreign currency f

## 3.2 Extract the Final RAG Sections

The section boundaries have been verified for both 10-K documents.

I will now extract the four sections selected for the common RAG baseline:

1. Business
2. Risk Factors
3. Management's Discussion and Analysis (MD&A)
4. Financial Statements and related notes

Each section will end at the next relevant 10-K boundary so that unrelated sections are not included.

In [23]:
def extract_rag_sections(
    text,
    section_occurrences,
    financial_override=None
):
    sections = {}
    text_lower = text.lower()

    # Find verified starting positions
    business_start = find_actual_section_start(
        text,
        section_patterns["Business"],
        section_occurrences["Business"]
    )

    risk_start = find_actual_section_start(
        text,
        section_patterns["Risk Factors"],
        section_occurrences["Risk Factors"]
    )

    mda_start = find_actual_section_start(
        text,
        section_patterns["MD&A"],
        section_occurrences["MD&A"]
    )

    financial_start = find_actual_section_start(
        text,
        section_patterns["Financial Statements"],
        section_occurrences["Financial Statements"]
    )

    # Find boundaries AFTER each actual section start
    risk_boundary = text_lower.find(
        "item 1b.",
        risk_start
    )

    mda_boundary = text_lower.find(
        "item 7a.",
        mda_start
    )

    financial_boundary = text_lower.find(
        "item 9.",
        financial_start
    )

    # Business ends where Risk Factors begins
    sections["Business"] = text[
        business_start:risk_start
    ].strip()

    # Risk Factors ends at Item 1B
    sections["Risk Factors"] = text[
        risk_start:risk_boundary
    ].strip()

    # MD&A ends at Item 7A
    sections["MD&A"] = text[
        mda_start:mda_boundary
    ].strip()

    # Financial Statements
    if financial_override is not None:
        financial_start, financial_end = financial_override

        sections["Financial Statements"] = text[
            financial_start:financial_end
        ].strip()

    else:
        sections["Financial Statements"] = text[
            financial_start:financial_boundary
        ].strip()

    return sections

# Netflix specific

In [24]:
# Netflix stores the actual financial statements later in the filing

netflix_financial_start = netflix_clean.lower().find(
    "netflix, inc. index to financial statements"
)

netflix_financial_end = netflix_clean.lower().find(
    "signatures",
    netflix_financial_start
)

In [25]:
netflix_rag_sections = extract_rag_sections(
    netflix_clean,
    section_occurrences["Netflix"],
    financial_override=(
        netflix_financial_start,
        netflix_financial_end
    )
)

cc_rag_sections = extract_rag_sections(
    cc_clean,
    section_occurrences["Coca Cola"]
)

---
# 4. Validate the Final RAG Sections

The four RAG sections have now been extracted using the verified 10-K boundaries.

Before chunking the text, I will perform a final boundary check to make sure sections that were intentionally excluded are not present inside the extracted content.

In [26]:
checks = {
    "Risk Factors": ["ITEM 1B.", "ITEM 1C."],
    "MD&A": ["ITEM 7A."],
    "Financial Statements": ["ITEM 9A.", "ITEM 9B.", "ITEM 9C."]
}

for name, sections in [
    ("Netflix", netflix_rag_sections),
    ("Coca Cola", cc_rag_sections)
]:
    print(f"\n========== {name} ==========")

    for section, unwanted_headers in checks.items():

        text = sections[section]

        print(f"\n{section}")

        for header in unwanted_headers:
            found = header.lower() in text.lower()

            print(f"  {header:<10} {'FOUND' if found else 'not found'}")


========== Netflix ==========

Risk Factors
  ITEM 1B.   not found
  ITEM 1C.   not found

MD&A
  ITEM 7A.   not found

Financial Statements
  ITEM 9A.   not found
  ITEM 9B.   not found
  ITEM 9C.   not found

========== Coca Cola ==========

Risk Factors
  ITEM 1B.   not found
  ITEM 1C.   not found

MD&A
  ITEM 7A.   not found

Financial Statements
  ITEM 9A.   not found
  ITEM 9B.   not found
  ITEM 9C.   not found


---
# 5. Create the RAG Chunks

The four extracted 10-K sections are still too large to retrieve efficiently as complete sections.

I will divide each section into smaller chunks using recursive character-based splitting.

Baseline configuration:

- Chunk size: 3,000 characters
- Chunk overlap: 300 characters

The chunks will later be converted into embeddings and stored in the vector database.

In [27]:
import re

CHUNK_SIZE = 2500
OVERLAP_SENTENCES = 2


def split_into_sentences(text):
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Split only after sentence-ending punctuation
    sentences = re.split(
        r'(?<=[.!?])\s+(?=[A-Z“"])',
        text
    )

    return sentences


def sentence_aware_chunks(text):
    sentences = split_into_sentences(text)

    chunks = []
    current_chunk = []

    for sentence in sentences:

        candidate = " ".join(current_chunk + [sentence])

        if len(candidate) <= CHUNK_SIZE:
            current_chunk.append(sentence)

        else:
            if current_chunk:
                chunks.append(" ".join(current_chunk))

            # Keep last 2 complete sentences as overlap
            current_chunk = (
                current_chunk[-OVERLAP_SENTENCES:]
                + [sentence]
            )

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

## 5.1 Split the Extracted Sections into Chunks

Each extracted section will now be divided into smaller chunks using the baseline text splitter.

The company and section information will be preserved for every chunk so that we can attach complete metadata before storing the chunks in the vector database.

In [28]:
def create_chunks(company, sections):
    chunks = []

    for section, text in sections.items():
        section_chunks = sentence_aware_chunks(text)

        for index, chunk in enumerate(section_chunks, start=1):
            chunks.append({
                "company": company,
                "section": section,
                "chunk_number": index,
                "text": chunk
            })

    return chunks

netflix_chunks = create_chunks("Netflix", netflix_rag_sections)

cc_chunks = create_chunks("Coca Cola", cc_rag_sections)

print("Netflix chunks:", len(netflix_chunks))
print("Coca Cola chunks:", len(cc_chunks))
print("Total chunks:", len(netflix_chunks) + len(cc_chunks))

Netflix chunks: 135
Coca Cola chunks: 270
Total chunks: 405


### 5.1.2 Inspect the Generated Chunks

The two companies produced 322 chunks using the baseline chunking configuration.

Before creating embeddings, I will inspect representative chunks from different sections to verify that the chunks preserve meaningful context and do not contain obvious boundary problems.

In [29]:
# Inspect representative chunks from both companies
for name, chunks in [
    ("Netflix", netflix_chunks),
    ("Coca Cola", cc_chunks)
]:
    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")

    # Inspect the first chunk and a middle chunk
    sample_indexes = [0, len(chunks) // 2]

    for index in sample_indexes:
        chunk = chunks[index]

        print(f"\n--- Chunk {index + 1} ---")
        print("Section:", chunk["section"])
        print("Characters:", len(chunk["text"]))
        print("\nText:")
        print(chunk["text"][:1500])


Netflix

--- Chunk 1 ---
Section: Business
Characters: 2207

Text:
Item 1. Business ABOUT US Netflix, Inc. (“Netflix”, the “Company”, “registrant”, “we”, or “us”) is one of the world's leading entertainment services offering TV series, films, games and live programming across a wide variety of genres and languages. Members can play, pause and resume watching as much as they want, anytime, anywhere, and can change their plans at any time. Our core strategy is to grow our business globally within the parameters of our operating margin target. We strive to continuously improve our members' experience by offering compelling content that delights them and attracts new members. We aim to offer a range of pricing plans, including our ad-supported subscription plan, to meet a variety of consumer needs. We seek to drive conversation around our content to further enhance member joy, and we are continuously enhancing our user interface to help our members more easily choose content that they wil

---
# 6. Add Standardized Metadata

Each RAG chunk will be associated with standardized metadata identifying its company, filing, section, and source.

The same metadata structure will be used for both companies so that the resulting RAG data can later be combined with the other six companies in the project.

*Example metadata structure:*

| Field	|netflix example|
|-------- |--------------|
| company	| Netflix |
| ticker	| NFLX |
| filing_type	| 10-K |
| filing_year	| 2025 |
| section	| MD&A |
| chunk_id	| NFLX_MDA_075 |
| source | SEC EDGAR |

> *The `chunk_id` is generated automatically.*
---


In [30]:
# Standard metadata for the two companies
company_metadata = {
    "Netflix": {
        "ticker": "NFLX",
        "filing_type": "10-K",
        "filing_year": 2025,
        "source": "SEC EDGAR"
    },
    "Coca Cola": {
        "ticker": "KO",
        "filing_type": "10-K",
        "filing_year": 2025,
        "source": "SEC EDGAR"
    }
}

def add_metadata(chunks):
    final_chunks = []

    for chunk in chunks:
        company = chunk["company"]
        section = chunk["section"]
        chunk_number = chunk["chunk_number"]

        metadata = company_metadata[company]

        final_chunks.append({
            "text": chunk["text"],
            "metadata": {
                "company": company,
                "ticker": metadata["ticker"],
                "filing_type": metadata["filing_type"],
                "filing_year": metadata["filing_year"],
                "section": section,
                "chunk_id": f"{metadata['ticker']}_{section.replace(' ', '_')}_{chunk_number:03d}",
                "source": metadata["source"]
            }
        })

    return final_chunks

netflix_final_chunks = add_metadata(netflix_chunks)
cc_final_chunks = add_metadata(cc_chunks)

all_chunks = netflix_final_chunks + cc_final_chunks

print("Netflix chunks:", len(netflix_final_chunks))
print("Coca Cola chunks:", len(cc_final_chunks))
print("Total chunks:", len(all_chunks))

Netflix chunks: 135
Coca Cola chunks: 270
Total chunks: 405


## 6.1 Inspect a Final RAG Chunk Record

The chunk text and standardized metadata have now been combined into the structure that will later be stored in the vector database.

I will inspect one complete record to verify that the text and metadata are correctly associated.

In [31]:
# Inspect one complete chunk record
sample_chunk = all_chunks[0]

print("Text:")
print(sample_chunk["text"][:1000])

print("\nMetadata:")

for key, value in sample_chunk["metadata"].items():
    print(f"{key}: {value}")

Text:
Item 1. Business ABOUT US Netflix, Inc. (“Netflix”, the “Company”, “registrant”, “we”, or “us”) is one of the world's leading entertainment services offering TV series, films, games and live programming across a wide variety of genres and languages. Members can play, pause and resume watching as much as they want, anytime, anywhere, and can change their plans at any time. Our core strategy is to grow our business globally within the parameters of our operating margin target. We strive to continuously improve our members' experience by offering compelling content that delights them and attracts new members. We aim to offer a range of pricing plans, including our ad-supported subscription plan, to meet a variety of consumer needs. We seek to drive conversation around our content to further enhance member joy, and we are continuously enhancing our user interface to help our members more easily choose content that they will find enjoyable. BUSINESS SEGMENTS We operate as one operatin

---
# 7. Validate RAG Chunk Metadata

Before creating embeddings, verify that every chunk contains the required metadata fields.

In [32]:
required_metadata = [
    "company",
    "ticker",
    "filing_type",
    "filing_year",
    "section",
    "chunk_id",
    "source"
]

for index, chunk in enumerate(all_chunks):
    metadata = chunk["metadata"]

    missing = [key for key in required_metadata if key not in metadata]

    if missing:
        print(f"Chunk {index + 1} is missing: {missing}")

print("Metadata validation complete.")

Metadata validation complete.


---

# Checkpoint

An embedding is a numerical representation of text meaning that allows us to compare a user's question with document chunks by semantic similarity.

**The important distinction**:

> A **smaller cosine distance** means more similar.

> A **larger cosine similarity** means more similar.

For cosine similarity:

$$similarity=\frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\|\|\mathbf{B}\|}$$

---



# 8. Load the Embedding Model

The embedding model converts text into numerical vectors that represent semantic meaning.

For the RAG baseline, we use `all-MiniLM-L6-v2`.

In [33]:
from google.colab import userdata
from huggingface_hub import login

try:
    # Attempt to retrieve token
    hf_token = userdata.get('HF_TOKEN')

    # Log into the Hugging Face Hub using the retrieved token
    login(hf_token)
    print("Authentication successful.")
except userdata.SecretNotFoundError:
    # If the token isn't found, smoothly continue without crashing
    print("Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.")

Authentication successful.


In [34]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully


## 8.1 Create a Test Embedding

Convert one RAG chunk into an embedding vector to verify that the embedding model is working correctly.

In [35]:
sample_text = all_chunks[0]["text"]

sample_embedding = embedding_model.encode(sample_text)

print("Embedding type:", type(sample_embedding))
print("Embedding dimensions:", len(sample_embedding))
print("First 10 values:", sample_embedding[:10])

Embedding type: <class 'numpy.ndarray'>
Embedding dimensions: 384
First 10 values: [ 0.00563927 -0.14448737 -0.03526784 -0.08091576  0.05348158  0.08532542
  0.10979541 -0.01008355  0.07611894 -0.01515785]


### 8.1.1 Test Semantic Similarity

Compare the embedding of a RAG document chunk with embeddings of related and unrelated questions using cosine similarity.

In [36]:
from sklearn.metrics.pairwise import cosine_similarity

related_question = "What does Netflix do?"
unrelated_question = "What are the symptoms of influenza?"

related_embedding = embedding_model.encode(related_question)
unrelated_embedding = embedding_model.encode(unrelated_question)

related_similarity = cosine_similarity(
    [sample_embedding],
    [related_embedding]
)[0][0]

unrelated_similarity = cosine_similarity(
    [sample_embedding],
    [unrelated_embedding]
)[0][0]

print("Related question similarity:", related_similarity)
print("Unrelated question similarity:", unrelated_similarity)

Related question similarity: 0.6025538
Unrelated question similarity: -0.011732353


---
# 9. Create Embeddings for All RAG Chunks

Convert all 322 RAG chunks into embedding vectors while keeping the original chunk order.

In [37]:
all_texts = [chunk["text"] for chunk in all_chunks]

all_embeddings = embedding_model.encode(all_texts, show_progress_bar=True)

print("Number of embeddings:", len(all_embeddings))
print("Embedding dimensions:", all_embeddings.shape[1])

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Number of embeddings: 405
Embedding dimensions: 384


## 9.1 Validate Chunk-Embedding Alignment

Verify that every RAG chunk has exactly one corresponding embedding vector.

In [38]:
print("Number of chunks:", len(all_chunks))
print("Number of embeddings:", len(all_embeddings))

assert len(all_chunks) == len(all_embeddings)

print("Chunk–embedding alignment verified.")

Number of chunks: 405
Number of embeddings: 405
Chunk–embedding alignment verified.


---
# 10. Set Up ChromaDB

ChromaDB will store the RAG embeddings, document text, and metadata so that relevant chunks can be retrieved later.

In [39]:
import chromadb

# Create a persistent ChromaDB client
chroma_client = chromadb.PersistentClient(
    path="./investiq_chroma_db"
)

# Create the InvestIQ collection
collection = chroma_client.get_or_create_collection(
    name="investiq_rag"
)

print("Collection name:", collection.name)
print("Documents currently stored:", collection.count())

Collection name: investiq_rag
Documents currently stored: 0


---
# 11. Create the RAG Collection

Create a ChromaDB collection that will store the `document embeddings`, `text`, and `metadata` for the **InvestIQ RAG system**.

In [40]:
chroma_client = chromadb.PersistentClient(path="/content/investiq_chroma_db")

collection = chroma_client.get_or_create_collection(name="investiq_rag")

print("Persistent ChromaDB collection ready.")
print("Collection name:", collection.name)

Persistent ChromaDB collection ready.
Collection name: investiq_rag


---
## 11.1 Check the ChromaDB Collection

Verify that the persistent RAG collection currently contains no documents before inserting the 322 chunks.

In [41]:
print("Collection name:", collection.name)
print("Number of records:", collection.count())

Collection name: investiq_rag
Number of records: 0


### 11.1.1 Prepare ChromaDB Records

Each ChromaDB record will contain:

- the chunk text as the document
- the embedding vector
- the standardized metadata
- a unique chunk ID

In [42]:
sample_record = all_chunks[0]

print("Document:")
print(sample_record["text"][:500])

print("\nEmbedding dimensions:")
print(len(all_embeddings[0]))

print("\nMetadata:")
print(sample_record["metadata"])

Document:
Item 1. Business ABOUT US Netflix, Inc. (“Netflix”, the “Company”, “registrant”, “we”, or “us”) is one of the world's leading entertainment services offering TV series, films, games and live programming across a wide variety of genres and languages. Members can play, pause and resume watching as much as they want, anytime, anywhere, and can change their plans at any time. Our core strategy is to grow our business globally within the parameters of our operating margin target. We strive to continu

Embedding dimensions:
384

Metadata:
{'company': 'Netflix', 'ticker': 'NFLX', 'filing_type': '10-K', 'filing_year': 2025, 'section': 'Business', 'chunk_id': 'NFLX_Business_001', 'source': 'SEC EDGAR'}


## 11.2 Store the RAG Chunks in ChromaDB

Store each chunk together with its embedding and metadata.

The `chunk ID` is used as the unique identifier for each ChromaDB record.

In [43]:
ids = [chunk["metadata"]["chunk_id"] for chunk in all_chunks]

documents = [chunk["text"] for chunk in all_chunks]

metadatas = [chunk["metadata"] for chunk in all_chunks]

collection.add(
    ids=ids,
    documents=documents,
    embeddings=all_embeddings.tolist(),
    metadatas=metadatas
)

print("Chunks stored successfully.")
print("Number of records:", collection.count())

Chunks stored successfully.
Number of records: 405


# 12. Test Semantic Retrieval

Before connecting the language model, test whether ChromaDB can retrieve relevant source chunks for a user question.

The question will first be converted into an embedding. ChromaDB will then compare that embedding with the stored chunk embeddings and return the most relevant chunks.

In [44]:
query = "What does Netflix do?"

query_embedding = embedding_model.encode(query).tolist()

# Retrieval
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Retrieved chunks:", len(results["documents"][0]))

for i, document in enumerate(results["documents"][0], start=1):
    print(f"\n========== Result {i} ==========")
    print("Document:")
    print(document[:1000])

    print("\nMetadata:")
    print(results["metadatas"][0][i - 1])

Retrieved chunks: 3

========== Result 1 ==========
Document:
Item 1. Business ABOUT US Netflix, Inc. (“Netflix”, the “Company”, “registrant”, “we”, or “us”) is one of the world's leading entertainment services offering TV series, films, games and live programming across a wide variety of genres and languages. Members can play, pause and resume watching as much as they want, anytime, anywhere, and can change their plans at any time. Our core strategy is to grow our business globally within the parameters of our operating margin target. We strive to continuously improve our members' experience by offering compelling content that delights them and attracts new members. We aim to offer a range of pricing plans, including our ad-supported subscription plan, to meet a variety of consumer needs. We seek to drive conversation around our content to further enhance member joy, and we are continuously enhancing our user interface to help our members more easily choose content that they will find

## 12.1 Inspect Retrieval Distances

Inspect the similarity distance returned by ChromaDB for each retrieved chunk.

This helps us understand how strongly each chunk matched the query.

In [45]:
print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n========== Result {i} ==========")
    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

Query: What does Netflix do?

========== Result 1 ==========
Distance: 0.7948923707008362
Company: Netflix
Section: Business
Chunk ID: NFLX_Business_001

========== Result 2 ==========
Distance: 1.159627079963684
Company: Netflix
Section: Financial Statements
Chunk ID: NFLX_Financial_Statements_004

========== Result 3 ==========
Distance: 1.1712496280670166
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_010


## 12.2 Test Retrieval for Risk Information

Test whether semantic retrieval can identify the Risk Factors section when the question concerns risks faced by Netflix.

In [46]:
query = "What risks does Netflix face?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n========== Result {i} ==========")
    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])
    print("\nDocument preview:")
    print(document[:500])

Query: What risks does Netflix face?

========== Result 1 ==========
Distance: 0.9038920998573303
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043

Document preview:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplated timeline or terms, or at all. Consummation of the WBD transaction is conditioned on, among other things, obtaining necessary governmental and regulatory approvals. If any of the conditions to the WBD transaction are not satisfied, it could delay or prevent the WBD transaction from occurring, which could result in Netflix's obligation to pay a $5.8 billion termination fee in certain 

========== Result 2 ==========
Distance: 0.9595227241516113
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_021

Document preview:
As a result of a dispute, we may have to develop non-infringing technology, enter into royalty or licensing agreements, adjust our content, merchandising or mar

### 12.2.1 Inspect a Stored Risk Factors Chunk

Verify that a Netflix Risk Factors chunk is present in ChromaDB and inspect its content and metadata.

In [47]:
risk_check = collection.get(ids=["NFLX_Risk_Factors_001"])

print("Retrieved records:", len(risk_check["documents"]))

print("\nDocument:")
print(risk_check["documents"][0][:1000])

print("\nMetadata:")
print(risk_check["metadatas"][0])

Retrieved records: 1

Document:
Item 1A. Risk Factors If any of the following risks actually occur, our business, financial condition and results of operations could be harmed. In that case, the trading price of our common stock could decline, and you could lose all or part of your investment. Risks Related to Our Business If our efforts to attract and retain members are not successful, our business will be adversely affected. We must continually add new members both to replace canceled memberships and to grow our business beyond our current membership base. Our ability to continue to attract and retain members will depend in part on our ability to consistently provide our members in countries around the globe with compelling content choices that keep our members engaged with our service, effectively drive conversation around our content and service, as well as provide a quality experience for choosing and enjoying TV series, films and games. Our penetration and growth rates have fluct

## 12.3 Test Retrieval with a Specific Risk Query

Use a more specific question that closely matches the content of the Risk Factors section.

This helps determine whether retrieval quality depends on query specificity.

In [48]:
query = "What risks does Netflix face when producing and licensing content?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n========== Result {i} ==========")
    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])
    print("\nDocument preview:")
    print(document[:500])

Query: What risks does Netflix face when producing and licensing content?

========== Result 1 ==========
Distance: 0.8271514773368835
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043

Document preview:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplated timeline or terms, or at all. Consummation of the WBD transaction is conditioned on, among other things, obtaining necessary governmental and regulatory approvals. If any of the conditions to the WBD transaction are not satisfied, it could delay or prevent the WBD transaction from occurring, which could result in Netflix's obligation to pay a $5.8 billion termination fee in certain 

========== Result 2 ==========
Distance: 0.8898700475692749
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_019

Document preview:
We have filed and we expect to file from time to time for trademark, copyright, and patent applications. Nevertheless, thes

# 13. Create a Retrieval Evaluation Set

Test semantic retrieval across the four main 10-K sections:

- Business
- Risk Factors
- MD&A
- Financial Statements

The expected section is used only to evaluate whether retrieval returned the appropriate source section.

In [49]:
retrieval_tests = [
    {
        "question": "What does Netflix do?",
        "expected_section": "Business"
    },
    {
        "question": "What risks could negatively affect Netflix's business and operations?",
        "expected_section": "Risk Factors"
    },
    {
        "question": "How did Netflix's revenue and operating income change compared with the previous year?",
        "expected_section": "MD&A"
    },
    {
        "question": "What were Netflix's total assets?",
        "expected_section": "Financial Statements"
    }
]

for test in retrieval_tests:
    print("Question:", test["question"])
    print("Expected section:", test["expected_section"])
    print()

Question: What does Netflix do?
Expected section: Business

Question: What risks could negatively affect Netflix's business and operations?
Expected section: Risk Factors

Question: How did Netflix's revenue and operating income change compared with the previous year?
Expected section: MD&A

Question: What were Netflix's total assets?
Expected section: Financial Statements



In [50]:
for test in retrieval_tests:
    query = test["question"]
    expected_section = test["expected_section"]

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    retrieved_sections = results["metadatas"][0]

    print("\n" + "=" * 70)
    print("Question:", query)
    print("Expected section:", expected_section)

    for i, metadata in enumerate(retrieved_sections, start=1):
        print(
            f"Result {i}: "
            f"{metadata['section']} | "
            f"{metadata['chunk_id']} | "
            f"Distance: {results['distances'][0][i-1]:.4f}"
        )

    found = any(
        metadata["section"] == expected_section
        for metadata in retrieved_sections
    )

    print("Expected section in top 3:", "YES ✅" if found else "NO ❌")


Question: What does Netflix do?
Expected section: Business
Result 1: Business | NFLX_Business_001 | Distance: 0.7949
Result 2: Financial Statements | NFLX_Financial_Statements_004 | Distance: 1.1596
Result 3: Risk Factors | NFLX_Risk_Factors_010 | Distance: 1.1712
Expected section in top 3: YES ✅

Question: What risks could negatively affect Netflix's business and operations?
Expected section: Risk Factors
Result 1: Risk Factors | NFLX_Risk_Factors_043 | Distance: 0.7199
Result 2: Risk Factors | NFLX_Risk_Factors_001 | Distance: 0.8189
Result 3: Financial Statements | NFLX_Financial_Statements_004 | Distance: 0.8455
Expected section in top 3: YES ✅

Question: How did Netflix's revenue and operating income change compared with the previous year?
Expected section: MD&A
Result 1: Financial Statements | NFLX_Financial_Statements_004 | Distance: 0.6250
Result 2: Financial Statements | NFLX_Financial_Statements_005 | Distance: 0.6797
Result 3: MD&A | NFLX_MD&A_002 | Distance: 0.7459
Expecte

## 13.1. Inspect Retrieved Evidence

Inspect the text retrieved for a financial question and verify that the relevant factual evidence is present in the retrieved chunk.

In [51]:
query = "What were Netflix's total assets?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:2000])

Query: What were Netflix's total assets?

Result 1
Distance: 0.7635542154312134
Company: Netflix
Section: Financial Statements
Chunk ID: NFLX_Financial_Statements_004

Retrieved text:
We also performed sensitivity analyses to evaluate the potential changes in the content amortization recorded that could result from changes in the assumptions. /s/ Ernst & Young LLP We have served as the Company's auditor since 2012. San Jose, California January 23, 2026 38 Table of Contents NETFLIX, INC. CONSOLIDATED STATEMENTS OF OPERATIONS (in thousands, except per share data) Year ended December 31, 2025 2024 2023 Revenues $ 45,183,036 $ 39,000,966 $ 33,723,297 Cost of revenues 23,275,329 21,038,464 19,715,368 Sales and marketing 3,301,306 2,917,554 2,657,883 Technology and development 3,391,390 2,925,295 2,675,758 General and administrative 1,888,408 1,702,039 1,720,285 Operating income 13,326,603 10,417,614 6,954,003 Other income (expense): Interest expense ( 776,510 ) ( 718,733 ) ( 699,826 ) Inter

## 13.2 Inspect Retrieved Risk Evidence

Inspect the retrieved chunks for a question about Netflix's reliance on U.S. government contracts.

Verify whether the retrieved Risk Factors chunk contains evidence relevant to the question.

In [52]:
query = "What risks could negatively affect Netflix's business and operations?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

Query: What risks could negatively affect Netflix's business and operations?

Result 1
Distance: 0.7198741436004639
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043

Retrieved text:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplated timeline or terms, or at all. Consummation of the WBD transaction is conditioned on, among other things, obtaining necessary governmental and regulatory approvals. If any of the conditions to the WBD transaction are not satisfied, it could delay or prevent the WBD transaction from occurring, which could result in Netflix's obligation to pay a $5.8 billion termination fee in certain specified circumstances. Further, as a condition to their approval of the WBD transaction, regulatory agencies may impose requirements, limitations or costs or require divestitures or place restrictions on the conduct of WBD's streaming and studios businesses after the closing. These requirement

## 13.3 Inspect the MD&A retrieval


In [53]:
query = "How did Netflix's revenue and operating income change compared with the previous year?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

Query: How did Netflix's revenue and operating income change compared with the previous year?

Result 1
Distance: 0.6250498294830322
Company: Netflix
Section: Financial Statements
Chunk ID: NFLX_Financial_Statements_004

Retrieved text:
We also performed sensitivity analyses to evaluate the potential changes in the content amortization recorded that could result from changes in the assumptions. /s/ Ernst & Young LLP We have served as the Company's auditor since 2012. San Jose, California January 23, 2026 38 Table of Contents NETFLIX, INC. CONSOLIDATED STATEMENTS OF OPERATIONS (in thousands, except per share data) Year ended December 31, 2025 2024 2023 Revenues $ 45,183,036 $ 39,000,966 $ 33,723,297 Cost of revenues 23,275,329 21,038,464 19,715,368 Sales and marketing 3,301,306 2,917,554 2,657,883 Technology and development 3,391,390 2,925,295 2,675,758 General and administrative 1,888,408 1,702,039 1,720,285 Operating income 13,326,603 10,417,614 6,954,003 Other income (expense): Inter

In [54]:
query = "According to Management’s Discussion and Analysis, what factors affected Netflix's financial performance?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

Query: According to Management’s Discussion and Analysis, what factors affected Netflix's financial performance?

Result 1
Distance: 0.6550902128219604
Company: Netflix
Section: Financial Statements
Chunk ID: NFLX_Financial_Statements_004

Retrieved text:
We also performed sensitivity analyses to evaluate the potential changes in the content amortization recorded that could result from changes in the assumptions. /s/ Ernst & Young LLP We have served as the Company's auditor since 2012. San Jose, California January 23, 2026 38 Table of Contents NETFLIX, INC. CONSOLIDATED STATEMENTS OF OPERATIONS (in thousands, except per share data) Year ended December 31, 2025 2024 2023 Revenues $ 45,183,036 $ 39,000,966 $ 33,723,297 Cost of revenues 23,275,329 21,038,464 19,715,368 Sales and marketing 3,301,306 2,917,554 2,657,883 Technology and development 3,391,390 2,925,295 2,675,758 General and administrative 1,888,408 1,702,039 1,720,285 Operating income 13,326,603 10,417,614 6,954,003 Other inco

# 14. Test Metadata-Filtered Retrieval

The previous retrieval tests showed that semantic search can retrieve financially relevant chunks, but it does not always retrieve the section requested by the question.

Because each chunk already contains section metadata, I will now restrict the search to the MD&A section and test whether this improves retrieval of MD&A content.

This experiment keeps the embedding model, chunking strategy, and ChromaDB collection unchanged.

In [55]:
query = "How did Netflix's revenue and operating income change compared with the previous year?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"section": "MD&A"}
)

print("Query:", query)
print("Filter: section = MD&A")

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

Query: How did Netflix's revenue and operating income change compared with the previous year?
Filter: section = MD&A

Result 1
Distance: 0.7458854913711548
Company: Netflix
Section: MD&A
Chunk ID: NFLX_MD&A_002

Retrieved text:
Operating margin for the year ended December 31, 2025 increased by approximately three percentage points as compared to the prior comparative period, primarily driven by the growth in revenues outpacing the growth in cost of revenues, sales and marketing, and general and administrative expenses. Net income for the year ended December 31, 2025 increased $2,270 million as compared to the prior comparative period, primarily due to a $2,909 million increase in operating income, driven by a $6,182 million increase in revenues and partially offset by a $2,237 million increase in cost of revenues primarily due to the increase in content amortization and other cost of revenues. The impact of higher operating income was partially offset by a $487 million increase in the 

## 14.1 Test Risk Factors Metadata Filtering

The previous experiment showed that filtering the ChromaDB search by the `section` metadata successfully retrieved relevant MD&A chunks.

I will now apply the same approach to the Risk Factors section and check whether the retrieved chunks contain relevant risk information.

In [56]:
query = "What risks could negatively affect Netflix's business and operations?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"section": "Risk Factors"}
)

print("Query:", query)
print("Filter: section = Risk Factors")

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

Query: What risks could negatively affect Netflix's business and operations?
Filter: section = Risk Factors

Result 1
Distance: 0.7198741436004639
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043

Retrieved text:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplated timeline or terms, or at all. Consummation of the WBD transaction is conditioned on, among other things, obtaining necessary governmental and regulatory approvals. If any of the conditions to the WBD transaction are not satisfied, it could delay or prevent the WBD transaction from occurring, which could result in Netflix's obligation to pay a $5.8 billion termination fee in certain specified circumstances. Further, as a condition to their approval of the WBD transaction, regulatory agencies may impose requirements, limitations or costs or require divestitures or place restrictions on the conduct of WBD's streaming and studios businesses after

# 15. Define the Retrieval Strategy

The retrieval experiments showed that unrestricted semantic search can retrieve relevant information, but semantically similar chunks from unrelated sections may rank higher than the section that is most appropriate for the question.

Metadata filtering can restrict retrieval to a relevant 10-K section and improve the relevance of the retrieved context.

The RAG system will therefore use two retrieval modes:

1. Metadata-filtered retrieval when the relevant section is known.
2. Unrestricted semantic retrieval when no specific section can be identified.

The retrieved chunks will then be passed to the language model as context for answer generation.

# 16. Create a Reusable Retrieval Function

The retrieval experiments above were performed using individual ChromaDB queries.

To reuse the same retrieval logic later in the RAG pipeline, I will create a function that supports both retrieval modes:

- filtered retrieval when a section is specified
- unrestricted retrieval when no section is specified

The function will return the retrieved documents, metadata, and distances.

In [57]:
def retrieve_chunks(query, n_results=3, section=None):
    query_embedding = embedding_model.encode(query).tolist()

    if section is not None:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where={"section": section}
        )
    else:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )

    retrieved_chunks = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved_chunks.append({
            "text": document,
            "metadata": metadata,
            "distance": distance
        })

    return retrieved_chunks

## 16.1 Test the Retrieval Function

I will test the reusable retrieval function using a Risk Factors question and restrict the search to the Risk Factors section.

The result should match the behavior of the earlier metadata-filtered retrieval experiment.

In [58]:
query = "What risks could negatively affect Netflix's business and operations?"

retrieved_chunks = retrieve_chunks(
    query=query,
    n_results=3,
    section="Risk Factors"
)

print("Number of retrieved chunks:", len(retrieved_chunks))

for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", chunk["distance"])
    print("Company:", chunk["metadata"]["company"])
    print("Section:", chunk["metadata"]["section"])
    print("Chunk ID:", chunk["metadata"]["chunk_id"])

    print("\nRetrieved text:")
    print(chunk["text"][:500])

Number of retrieved chunks: 3

Result 1
Distance: 0.7198741436004639
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043

Retrieved text:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplated timeline or terms, or at all. Consummation of the WBD transaction is conditioned on, among other things, obtaining necessary governmental and regulatory approvals. If any of the conditions to the WBD transaction are not satisfied, it could delay or prevent the WBD transaction from occurring, which could result in Netflix's obligation to pay a $5.8 billion termination fee in certain 

Result 2
Distance: 0.8189029097557068
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_001

Retrieved text:
Item 1A. Risk Factors If any of the following risks actually occur, our business, financial condition and results of operations could be harmed. In that case, the trading price of our common stock could decline, and

# 17. Build the Retrieved Context

The retrieval step gives us a list of relevant chunks.

Before sending these chunks to the language model, we need to combine their text into a single context block.

This context will later be provided to Qwen together with the user's question.

For now, I will only build and inspect the context. I will not generate an answer with Qwen yet.

In [59]:
context_parts = []

for i, chunk in enumerate(retrieved_chunks, start=1):
    context_parts.append(
        f"[Retrieved Source {i}]\n"
        f"Company: {chunk['metadata']['company']}\n"
        f"Section: {chunk['metadata']['section']}\n"
        f"Chunk ID: {chunk['metadata']['chunk_id']}\n"
        f"Text:\n{chunk['text']}"
    )

retrieved_context = "\n\n".join(context_parts)

print(retrieved_context[:5000])

[Retrieved Source 1]
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043
Text:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplated timeline or terms, or at all. Consummation of the WBD transaction is conditioned on, among other things, obtaining necessary governmental and regulatory approvals. If any of the conditions to the WBD transaction are not satisfied, it could delay or prevent the WBD transaction from occurring, which could result in Netflix's obligation to pay a $5.8 billion termination fee in certain specified circumstances. Further, as a condition to their approval of the WBD transaction, regulatory agencies may impose requirements, limitations or costs or require divestitures or place restrictions on the conduct of WBD's streaming and studios businesses after the closing. These requirements, limitations, costs, divestitures or restrictions could jeopardize or delay the consummation of the WBD 

# 18. Create a Reusable Context Builder

The retrieved chunks will eventually be passed to the language model.

To keep the RAG pipeline modular, I will create a reusable function that converts retrieved chunks into a formatted context string.

This function will preserve the source metadata together with the retrieved text.

In [60]:
def build_retrieved_context(retrieved_chunks):
    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"[Retrieved Source {i}]\n"
            f"Company: {chunk['metadata']['company']}\n"
            f"Section: {chunk['metadata']['section']}\n"
            f"Chunk ID: {chunk['metadata']['chunk_id']}\n"
            f"Text:\n{chunk['text']}"
        )

    return "\n\n".join(context_parts)

## 18.1 Test the Context Builder

I will use the retrieved chunks from the previous retrieval test and pass them through the new context-building function.

The resulting context should contain the retrieved source information and text in the same order as before.

In [61]:
retrieved_context = build_retrieved_context(retrieved_chunks)

print(retrieved_context[:5000])

[Retrieved Source 1]
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043
Text:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplated timeline or terms, or at all. Consummation of the WBD transaction is conditioned on, among other things, obtaining necessary governmental and regulatory approvals. If any of the conditions to the WBD transaction are not satisfied, it could delay or prevent the WBD transaction from occurring, which could result in Netflix's obligation to pay a $5.8 billion termination fee in certain specified circumstances. Further, as a condition to their approval of the WBD transaction, regulatory agencies may impose requirements, limitations or costs or require divestitures or place restrictions on the conduct of WBD's streaming and studios businesses after the closing. These requirements, limitations, costs, divestitures or restrictions could jeopardize or delay the consummation of the WBD 

# 19. Build the RAG Prompt

The RAG prompt combines the user's question with the retrieved source context.

The prompt instructs the language model to:

- answer using the retrieved context
- prioritize information supported by the provided sources
- avoid inventing information that is not supported by the context
- indicate when the retrieved information is insufficient
- identify the retrieved source used for the answer

At this stage, the prompt is only constructed and inspected. It will not yet be sent to Qwen.

In [62]:
question = "What risks could negatively affect Netflix's business and operations?"

rag_prompt = f"""
    You are an investment research assistant.

    Answer the user's question using the retrieved source context below.

    Rules:
    - Use the retrieved context as the primary source of evidence.
    - Do not invent facts that are not supported by the retrieved context.
    - If the retrieved context is insufficient to answer the question, say that the available information is insufficient.
    - Identify the source chunk IDs that support your answer.

    User question:
    {question}

    Retrieved context:
    {retrieved_context}
    """

print(rag_prompt[:6000])


    You are an investment research assistant.

    Answer the user's question using the retrieved source context below.

    Rules:
    - Use the retrieved context as the primary source of evidence.
    - Do not invent facts that are not supported by the retrieved context.
    - If the retrieved context is insufficient to answer the question, say that the available information is insufficient.
    - Identify the source chunk IDs that support your answer.

    User question:
    What risks could negatively affect Netflix's business and operations?

    Retrieved context:
    [Retrieved Source 1]
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043
Text:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplated timeline or terms, or at all. Consummation of the WBD transaction is conditioned on, among other things, obtaining necessary governmental and regulatory approvals. If any of the conditions to the WBD trans

# 20. Define the RAG Response Format

The RAG response should be structured so that the application can use the generated information consistently.

The model will be asked to return:

- an answer to the user's question
- the source chunk IDs supporting the answer
- an indication of whether the retrieved context was sufficient

This structure will later make it easier to display the answer and its sources in the Streamlit application.

In [63]:
rag_prompt = f"""
You are an investment research assistant.

Answer the user's question using the retrieved source context below.

Rules:
- Use the retrieved context as the primary source of evidence.
- Do not invent facts that are not supported by the retrieved context.
- If the retrieved context is insufficient to answer the question, say that the available information is insufficient.
- Only cite source chunk IDs that actually appear in the retrieved context.
- Return your response as valid JSON using exactly these fields:
  "answer"
  "sources"
  "sufficient_context"

The "sources" field must contain a list of source chunk IDs.
The "sufficient_context" field must be either true or false.

User question:
{question}

Retrieved context:
{retrieved_context}
"""

print(rag_prompt[:6000])


You are an investment research assistant.

Answer the user's question using the retrieved source context below.

Rules:
- Use the retrieved context as the primary source of evidence.
- Do not invent facts that are not supported by the retrieved context.
- If the retrieved context is insufficient to answer the question, say that the available information is insufficient.
- Only cite source chunk IDs that actually appear in the retrieved context.
- Return your response as valid JSON using exactly these fields:
  "answer"
  "sources"
  "sufficient_context"

The "sources" field must contain a list of source chunk IDs.
The "sufficient_context" field must be either true or false.

User question:
What risks could negatively affect Netflix's business and operations?

Retrieved context:
[Retrieved Source 1]
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043
Text:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplat

# 21. Load the Fine-Tuned Qwen Model

The RAG pipeline will use the fine-tuned InvestIQ Qwen adapter created during the fine-tuning stage.

The adapter is already saved in Google Drive, so the model does not need to be trained again.

I will first import the libraries required for inference and then reload the base Qwen model together with the fine-tuned adapter.

In [64]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 22. Reload the Fine-Tuned InvestIQ Model

The fine-tuned model consists of:

- the original Qwen2.5-1.5B-Instruct base model
- the InvestIQ LoRA adapter trained earlier

The adapter is stored separately, so I will load the base model first and then attach the adapter.

This makes the RAG notebook independent from the previous training notebook.

In [65]:
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

ADAPTER_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/week_04/outputs/Finetuned_version01/InvestIQ_finetuned_adapter_updated")

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    device_map="cuda",
    torch_dtype=torch.float16
)

base_model.config.use_cache = True

trained_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

trained_model.eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

# 23. Generate the RAG Answer

The fine-tuned InvestIQ model has now been successfully reloaded.

I will provide it with the RAG prompt containing:

- the user's question
- the retrieved 10-K evidence
- instructions for source-grounded JSON output

The generated response will be inspected before we make any changes to the pipeline.

In [66]:
messages = [
    {
        "role": "user",
        "content": rag_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(rag_response)

{"answer":"The risk factors listed describe several possible negative impacts on Netflix's business and operations.","sources":["NFLX_Risk_Factors_043","NFLX_Risk_Factors_001","NFLX_Risk_Factors_021"],"sufficient_context":true}


# 24. Test RAG Generation Without Structured JSON

The previous generation test required the model to produce a strict JSON response.

The model followed the requested structure but returned an empty answer.

To determine whether the problem is caused by the RAG context or by the structured-output instruction, I will temporarily remove the JSON requirement.

The model will instead be asked to answer the question directly using only the retrieved context.

This is a diagnostic experiment and does not change the final RAG design yet.

In [67]:
diagnostic_prompt = f"""
You are an investment research assistant.

Answer the user's question using only the retrieved source context below.

Rules:
- Use the retrieved context as the primary source of evidence.
- Do not invent facts that are not supported by the retrieved context.
- If the retrieved context is insufficient, clearly say so.
- Explain the relevant risks using the information provided in the sources.
- Mention the source chunk IDs that support your answer.

User question:
{question}

Retrieved context:
{retrieved_context}
"""

messages = [
    {
        "role": "user",
        "content": diagnostic_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

diagnostic_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(diagnostic_response)

{"summary":"The risk factors listed describe various negative impacts that could affect Netflix's business and operations.","risks":["The WBD transaction may fail to close on schedule or at all","Our ability to attract and retain members may be impacted due to competition and changing market conditions","Adverse macroeconomic conditions can harm our ability to attract and retain members","Disputes with content suppliers may require substantial resources to resolve","Cybersecurity breaches and system failures can compromise member privacy and service reliability"],"probability_interpretation":"","recommendation_context":"The investor should consider whether the company's revenue growth, profitability, cash flow, debt-to-equity ratio, historical volatility, and available compensation measures match its stated objectives and whether the company has sufficient internal controls and external protections.","response_type":"investment_analysis"}


## 24.2 Strengthen the RAG Grounding Instructions

The first generation test produced a useful answer but also included several claims that were not supported by the retrieved context.

I will therefore strengthen the grounding instructions.

The model must treat the retrieved context as the evidence boundary for the answer:

- factual claims must be supported by the retrieved context
- unsupported claims must be omitted
- the model should not fill missing information using its general knowledge
- source chunk IDs should correspond to the evidence used

This is a prompt-level change only. The model and retrieval system remain unchanged.

In [68]:
grounded_prompt = f"""
You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Grounding rules:
- Treat the retrieved context as the evidence boundary for your answer.
- Every factual claim in your answer must be supported by the retrieved context.
- Do not use your general knowledge to add facts that are not present in the retrieved context.
- If a detail is not supported by the retrieved context, leave it out.
- Do not reinterpret or contradict the retrieved source text.
- If the retrieved context is insufficient to answer part of the question, explicitly state that the available context is insufficient.
- After each major factual statement, identify the source chunk ID that supports it.
- Use only source chunk IDs that actually appear in the retrieved context.

User question:
{question}

Retrieved context:
{retrieved_context}
"""

print(grounded_prompt[:6000])


You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Grounding rules:
- Treat the retrieved context as the evidence boundary for your answer.
- Every factual claim in your answer must be supported by the retrieved context.
- Do not use your general knowledge to add facts that are not present in the retrieved context.
- If a detail is not supported by the retrieved context, leave it out.
- Do not reinterpret or contradict the retrieved source text.
- If the retrieved context is insufficient to answer part of the question, explicitly state that the available context is insufficient.
- After each major factual statement, identify the source chunk ID that supports it.
- Use only source chunk IDs that actually appear in the retrieved context.

User question:
What risks could negatively affect Netflix's business and operations?

Retrieved context:
[Retrieved Source 1]
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_

## 24.3 Test the Strengthened Grounding Prompt

I will now generate an answer using the strengthened grounding instructions.

The model, retrieved chunks, generation settings, and retrieval method remain unchanged.

Only the prompt instructions have changed.

This allows us to compare the result with the previous diagnostic generation.

In [69]:
messages = [
    {
        "role": "user",
        "content": grounded_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

grounded_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(grounded_response)

The risk factors listed above describe several possible negative impacts on Netflix's business and operations:

1. **WBD Transaction Risks**: The company faces uncertainty about whether the WBD transaction can proceed according to its original plan due to regulatory delays, missing conditions, or other obstacles. This could lead to financial penalties, delayed completion, or reduced expected benefits.

2. **Member Acquisition Risks**: The company relies on attracting new members but cannot guarantee this process. Member cancellations can harm revenue and retention. The company must compete with established players and potentially disruptive alternatives.

3. **Competitor Impact**: The company competes with various entertainment providers, some of which offer lower prices, broader content, or different targeted audiences. Negative market trends affecting these competitors could hurt Netflix.

4. **Regulatory and Compliance Risks**: Obtaining necessary approvals and complying with changi

## 24.4 Test a Simple Grounded Answer Format

The previous grounding prompt reduced unsupported claims, but the model did not include the requested source chunk IDs.

In this test, I will use a simpler output structure:

- Answer
- Supporting Sources

The retrieval method, retrieved chunks, model, and generation settings remain unchanged.

This helps test whether a simpler response format improves source attribution without introducing the stricter JSON format that previously caused an empty answer.

In [70]:
source_prompt = f"""
You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Grounding rules:
- Treat the retrieved context as the evidence boundary for your answer.
- Every factual claim must be supported by the retrieved context.
- Do not add facts from your general knowledge.
- If a detail is not supported by the retrieved context, leave it out.
- Do not reinterpret or contradict the retrieved source text.
- If the context is insufficient, clearly state that.

Use exactly this response structure:

Answer:
<Give a concise answer supported only by the retrieved context.>

Supporting Sources:
<List only the chunk IDs that directly support the answer.>

User question:
{question}

Retrieved context:
{retrieved_context}
"""

print(source_prompt[:6000])


You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Grounding rules:
- Treat the retrieved context as the evidence boundary for your answer.
- Every factual claim must be supported by the retrieved context.
- Do not add facts from your general knowledge.
- If a detail is not supported by the retrieved context, leave it out.
- Do not reinterpret or contradict the retrieved source text.
- If the context is insufficient, clearly state that.

Use exactly this response structure:

Answer:
<Give a concise answer supported only by the retrieved context.>

Supporting Sources:
<List only the chunk IDs that directly support the answer.>

User question:
What risks could negatively affect Netflix's business and operations?

Retrieved context:
[Retrieved Source 1]
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043
Text:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the cu

# 25. Generate a Grounded Answer with Fine-Tuned Qwen

The previous step constructed and inspected the prompt but did not run the language model.

In this step, the prompt will be tokenized and passed to the **fine-tuned Qwen model for inference**.

Inference means using an already-trained model to generate a prediction or response for new input. The model weights are not updated during this process.

The generation settings remain the same as in the previous grounding experiment so that the results can be compared fairly.

In [71]:
messages = [
    {
        "role": "user",
        "content": source_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

source_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(source_response)

The risks described in the retrieved context include delays or cancellations due to regulatory approvals, difficulties in integrating acquired businesses, competition from various entertainment providers, changing consumer preferences, and cybersecurity-related disruptions. The exact nature of each risk can change over time, so the investor should review the most recent available information before making a decision.


---
## 26.1 Fine-Tuned Qwen Without RAG
---

In [72]:
no_rag_prompt = f"""
You are an investment research assistant.

Answer the user's question as accurately as possible.

User question:
{question}
"""

messages = [
    {
        "role": "user",
        "content": no_rag_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

no_rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(no_rag_response)

{"summary":"The main risk is that streaming services can become less popular over time due to changing consumer preferences or competition.","key_factors":["Streaming service popularity","Consumer preference changes","Competition from other entertainment providers"],"risks":["Declining subscriber growth","Changing viewing habits","Increased competition from new entrants","Decreased advertising revenue","Financial losses"],"probability_interpretation":"","recommendation_context":"Netflix should monitor its audience trends, evaluate the impact of new competitors, and consider whether its content strategy remains relevant.","response_type":"investment_analysis"}


In [73]:
probability_question = """
The model estimates a 72% probability of a positive 1-year return for a company.
The company also has declining revenue growth and increasing debt.
How should I interpret these signals together?
"""

probability_prompt = f"""
You are an investment research assistant.

Answer the user's question as accurately as possible.

User question:
{probability_question}
"""

messages = [
    {"role": "user", "content": probability_prompt}
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

probability_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(probability_response)

{"summary":"A 72% estimated probability is stronger evidence that the model views the combined profile as relatively favorable than the separate indicators themselves.","key_factors":["Model-estimated probability","Revenue growth trend","Debt trend","Historical market conditions"],"risks":["The model estimate does not prove that the future outcome will be positive","Declining growth may indicate weakening business fundamentals","Increasing leverage can increase financial risk","Past performance does not guarantee future returns"],"probability_interpretation":"The 72% probability indicates that the model estimates a greater-than-even likelihood of a positive 1-year return under its binary target definition. It does not mean that every dollar invested would yield exactly 72% profit.","recommendation_context":"The investor should consider whether the model signal remains consistent with the company's other financial trends before interpreting them together.","response_type":"investment_an

---
## 26.2. Generate an Answer With RAG
---

The previous experiment generated an answer using the fine-tuned Qwen model without external retrieved context.

Now I will provide the same model with relevant chunks retrieved from the InvestIQ ChromaDB collection.

The goal is to observe how the retrieved 10-K evidence changes the answer.

The comparison uses:

- the same fine-tuned Qwen model
- the same user question
- the same generation settings
- retrieved evidence from the Lockheed Martin 2025 10-K

In [75]:
question = "What risks does Netflix face when producing and licensing content?"

retrieved_chunks = retrieve_chunks(
    question,
    n_results=3,
    section="Risk Factors"
)

retrieved_context = build_retrieved_context(retrieved_chunks)

print(retrieved_context)

[Retrieved Source 1]
Company: Netflix
Section: Risk Factors
Chunk ID: NFLX_Risk_Factors_043
Text:
Risk Factors Related to the WBD Transaction The WBD transaction may not be completed on the currently contemplated timeline or terms, or at all. Consummation of the WBD transaction is conditioned on, among other things, obtaining necessary governmental and regulatory approvals. If any of the conditions to the WBD transaction are not satisfied, it could delay or prevent the WBD transaction from occurring, which could result in Netflix's obligation to pay a $5.8 billion termination fee in certain specified circumstances. Further, as a condition to their approval of the WBD transaction, regulatory agencies may impose requirements, limitations or costs or require divestitures or place restrictions on the conduct of WBD's streaming and studios businesses after the closing. These requirements, limitations, costs, divestitures or restrictions could jeopardize or delay the consummation of the WBD 

## 26.3 Generate a Grounded RAG Answer

The retrieved Risk Factors chunks provide evidence about Lockheed Martin's dependence on U.S. Government contracts, procurement requirements, regulatory compliance, and changes in government contracting policies.

I will now provide these retrieved chunks to the fine-tuned Qwen model and ask it to answer the original question using the evidence.

The model must treat the retrieved context as the evidence boundary and should not add unsupported facts.

In [76]:
rag_prompt = f"""
You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Grounding rules:
- Treat the retrieved context as the evidence boundary for your answer.
- Every factual claim must be supported by the retrieved context.
- Do not add facts from your general knowledge.
- Focus only on the company asked about in the user's question.
- Ignore retrieved sources that refer to other companies.
- Do not compare companies unless the user explicitly asks for a comparison.
- Do not invent relationships between companies.
- If a detail is not supported by the relevant retrieved context, leave it out.
- Do not contradict or reinterpret the source text.
- Mention the source chunk IDs that support your answer.

User question:
{question}

Retrieved context:
{retrieved_context}
"""

messages = [
    {"role": "user", "content": rag_prompt}
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(rag_response)

The risk factors mentioned include:

- The WBD transaction may not proceed according to plan due to delays, lack of required approvals, or failing to meet contractual obligations.
- Regulatory approvals may impose additional conditions or costs.
- Third-party challenges to intellectual property rights could harm the company's brand and marketability.
- Integration difficulties may delay achieving expected synergies.
- Domain name disputes could damage the company's reputation.
- Intellectual property claims could be expensive and limit the company's ability to defend its rights.
- Content availability may be limited if rights holders refuse to grant necessary permissions.
- Programming costs may rise as competition intensifies.
- A diverse content mix may be needed to retain members, but maintaining exclusivity can increase costs.
- Obtaining new content may involve lengthy negotiation processes and uncertainty about the final terms.


## 26.4 Compare RAG and Non-RAG Responses

The same fine-tuned Qwen model was asked the same question in two conditions:

**1. Without RAG** — Qwen answered using its learned behavior and existing model knowledge.

**2. With RAG** — Qwen received three relevant Lockheed Martin Risk Factors chunks from the 2025 10-K.

I will compare the responses based on:

- use of company-specific evidence
- factual grounding
- specificity
- source attribution
- unsupported claims
- response structure

In [77]:
print("=" * 80)
print("WITHOUT RAG")
print("=" * 80)
print(no_rag_response)

print("\n" + "=" * 80)
print("WITH RAG")
print("=" * 80)
print(rag_response)

WITHOUT RAG
{"summary":"The main risk is that streaming services can become less popular over time due to changing consumer preferences or competition.","key_factors":["Streaming service popularity","Consumer preference changes","Competition from other entertainment providers"],"risks":["Declining subscriber growth","Changing viewing habits","Increased competition from new entrants","Decreased advertising revenue","Financial losses"],"probability_interpretation":"","recommendation_context":"Netflix should monitor its audience trends, evaluate the impact of new competitors, and consider whether its content strategy remains relevant.","response_type":"investment_analysis"}

WITH RAG
The risk factors mentioned include:

- The WBD transaction may not proceed according to plan due to delays, lack of required approvals, or failing to meet contractual obligations.
- Regulatory approvals may impose additional conditions or costs.
- Third-party challenges to intellectual property rights could h

# 27. Test Structured RAG Output

The previous RAG response used the retrieved evidence successfully, but it did not reliably return the requested source chunk IDs.

For the final InvestIQ application, structured output would make it easier for Streamlit to display:

- the generated answer
- the supporting source chunks
- whether sufficient evidence was retrieved

I will therefore test whether the fine-tuned model can follow a simple JSON response structure when RAG context is provided.

In [78]:
structured_rag_prompt = f"""
You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Rules:
- Treat the retrieved context as the evidence boundary.
- Every factual claim must be supported by the retrieved context.
- Do not add facts from general knowledge.
- If the context does not support a detail, leave that detail out.
- Do not invent source IDs.
- Only use source IDs that appear in the retrieved context.

Return ONLY valid JSON using exactly this structure:

{{
  "answer": "A concise answer supported by the retrieved context.",
  "sources": ["source_id_1", "source_id_2"]
}}

User question:
{question}

Retrieved context:
{retrieved_context}
"""

messages = [
    {"role": "user", "content": structured_rag_prompt}
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

structured_rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(structured_rag_response)

{"answer":"Netflix faces several risks when producing and licensing content, including delays or failures in completing transactions, uncertainty regarding regulatory approvals, potential conflicts with third-party intellectual property rights, difficulties in integrating acquired businesses, and changes in market conditions affecting its ability to secure desired rights.","sources":["NFLX_Risk_Factors_043","NFLX_Risk_Factors_019","NFLX_Risk_Factors_017"]}


In [79]:
# @title
import json

# Retrieve all stored chunks and metadata from ChromaDB
data = collection.get(
    include=["documents", "metadatas"]
)

processed_chunks = []

for doc_id, document, metadata in zip(
    data["ids"],
    data["documents"],
    data["metadatas"]
):
    processed_chunks.append({
        "id": doc_id,
        "text": document,
        "metadata": metadata
    })

# Save to JSON
output_path = "/content/olga_processed_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        processed_chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Export complete.")
print("Total chunks:", len(processed_chunks))
print("Saved to:", output_path)

Export complete.
Total chunks: 405
Saved to: /content/olga_processed_chunks.json
